# Paso 1: Generación de Datos Sintéticos

## ¿Qué son los datos sintéticos?

Los **datos sintéticos** son datos generados artificialmente por ordenador que imitan las características de datos reales. En lugar de utilizar información de pacientes reales (lo cual requeriría permisos y plantearía problemas de privacidad), creamos un conjunto de datos ficticio que sigue los mismos patrones estadísticos que encontraríamos en una clínica real.

## ¿Por qué los usamos aquí?

- **Privacidad**: No necesitamos datos de pacientes reales para desarrollar y probar el sistema.
- **Control**: Podemos definir exactamente cuántos deportistas queremos y qué características tendrán.
- **Reproducibilidad**: Con la misma semilla aleatoria, siempre obtenemos exactamente los mismos datos.
- **Desarrollo**: Nos permiten construir y validar el modelo antes de aplicarlo con datos reales.

Este notebook genera el conjunto de datos base que usaremos en todos los pasos siguientes.

In [1]:
import sys
from pathlib import Path

# Añadir el directorio raíz del proyecto al path
proyecto_raiz = Path("..").resolve()
sys.path.insert(0, str(proyecto_raiz))

from src.generador_datos import generar_dataset
from src.variables import VARIABLES, TOTAL_COLUMNAS
import pandas as pd

## Configuración

Aquí puedes ajustar dos parámetros:

| Parámetro | Descripción | Valor por defecto |
|-----------|-------------|-------------------|
| `N_DEPORTISTAS` | Número de deportistas que tendrá el dataset | 500 |
| `SEMILLA` | Número que controla la aleatoriedad (mismo número = mismos datos siempre) | 42 |

**Nota:** Este notebook genera un dataset de exploración (500 deportistas). El modelo de producción se entrena en `03_entrenamiento_modelo.ipynb` con 5 semillas × 1 000 muestras = 4 826 deportistas.

In [2]:
N_DEPORTISTAS = 500
SEMILLA = 42

df = generar_dataset(n_deportistas=N_DEPORTISTAS, semilla=SEMILLA)
print(f"Dataset generado: {df.shape[0]} deportistas, {df.shape[1]} columnas")

Generando dataset sintético v2.3 con 500 deportistas (semilla=42)...
  [1/5] Bloque contexto...
  [2/5] Bloque fuerza...
  [3/5] Bloque movilidad...
  [4/5] Bloque control...
  [5/5] Inyectando casos frontera...
  Aplicando reglas v2.3 y calculando score de confianza...

Distribución de riesgo:
  bajo            :  28.6 %
  medio           :  25.0 %
  alto            :  42.4 %
  no_concluyente  :   4.0 %

Distribución de confianza:
  alta            :  21.2 %
  media           :  38.6 %
  baja            :  40.2 %

Dataset generado: 500 filas × 36 columnas.

Dataset generado: 500 deportistas, 36 columnas


## Vista previa de los datos

A continuación se muestran los primeros 10 deportistas del dataset. Puedes ver todas las variables que se han generado para cada uno.

In [3]:
df.head(10)

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,peso_corporal,nivel_actividad,historial_lesional,dolor_percibido_nrs,hooper_index,riesgo_lesion,score_total,confianza_score,confianza_categoria,reglas_activadas
0,197.8,158.4,117.2,99.6,98.1,89.6,46.2,63.9,85.3,68.6,...,61.6,activo,1,0,10,alto,22.5,52.0,baja,A1_izq
1,304.1,329.8,258.2,271.4,136.2,195.0,119.7,132.2,229.8,159.1,...,62.7,elite,0,0,14,medio,18.0,64.0,media,M7
2,154.9,82.2,153.6,127.0,46.0,67.3,29.8,43.0,81.5,70.0,...,50.5,activo,2,1,17,alto,31.5,44.0,baja,A2_der
3,177.0,336.4,198.6,231.4,133.0,103.8,73.5,90.7,184.8,125.6,...,70.7,activo,3,1,10,bajo,11.5,80.0,alta,—
4,160.9,230.5,224.4,224.7,214.5,200.5,112.7,56.4,222.6,184.1,...,80.3,recreacional,0,0,14,medio,18.0,68.0,media,—
5,180.7,120.9,133.6,175.7,77.4,76.6,62.0,61.6,140.7,118.5,...,75.1,recreacional,2,1,10,alto,28.0,48.0,baja,A1_der
6,145.5,179.6,140.8,131.3,69.2,111.9,47.4,62.3,77.8,86.8,...,56.6,activo,0,0,12,medio,23.5,56.0,baja,—
7,228.6,242.8,120.6,134.5,91.8,73.9,76.6,64.5,181.0,182.9,...,65.4,recreacional,1,0,11,bajo,16.5,64.0,media,—
8,431.3,306.2,278.9,188.6,213.5,201.4,99.7,115.6,188.5,289.2,...,85.4,recreacional,2,0,15,alto,17.0,64.0,media,A1_izq
9,368.1,177.0,197.6,150.0,190.2,154.6,79.4,75.6,170.2,238.8,...,81.7,recreacional,0,8,17,no_concluyente,14.0,72.0,media,A1_izq


## Distribución de niveles de riesgo

Una de las columnas más importantes del dataset es el **nivel de riesgo de lesión**. Aquí podemos ver cuántos deportistas caen en cada categoría.

In [4]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Tabla de frecuencias
print("Distribución de niveles de riesgo:")
print("="*40)
distribucion = df["riesgo_lesion"].value_counts().sort_index()
for nivel, cantidad in distribucion.items():
    porcentaje = cantidad / len(df) * 100
    print(f"  {nivel}: {cantidad} deportistas ({porcentaje:.1f}%)")
print("="*40)

# Gráfico de barras
fig, ax = plt.subplots(figsize=(8, 5))

colores = {"Bajo": "#2ecc71", "Medio": "#f39c12", "Alto": "#e74c3c"}
niveles = distribucion.index.tolist()
valores = distribucion.values.tolist()
barras_colores = [colores.get(n, "#3498db") for n in niveles]

barras = ax.bar(niveles, valores, color=barras_colores, edgecolor="white", linewidth=1.5)

# Etiquetas encima de cada barra
for barra, valor in zip(barras, valores):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 5,
        str(valor),
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.set_title("Distribución de niveles de riesgo de lesión", fontsize=14, pad=15)
ax.set_xlabel("Nivel de riesgo", fontsize=12)
ax.set_ylabel("Número de deportistas", fontsize=12)
ax.set_ylim(0, max(valores) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
ruta_fig = proyecto_raiz / "figuras" / "distribucion_riesgo.png"
plt.savefig(ruta_fig, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Figura guardada en: {ruta_fig}")

Distribución de niveles de riesgo:
  alto: 212 deportistas (42.4%)
  bajo: 143 deportistas (28.6%)
  medio: 125 deportistas (25.0%)
  no_concluyente: 20 deportistas (4.0%)
Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/distribucion_riesgo.png


## Estadísticas descriptivas

La tabla siguiente muestra un resumen estadístico de todas las variables numéricas del dataset:

- **count**: número de valores disponibles
- **mean**: media (promedio)
- **std**: desviación estándar (dispersión de los datos)
- **min / max**: valores mínimo y máximo
- **25% / 50% / 75%**: percentiles (el 50% es la mediana)

In [5]:
df.describe().round(2)

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,single_leg_squat_valgo_izq,single_leg_hop_der,single_leg_hop_izq,edad,peso_corporal,historial_lesional,dolor_percibido_nrs,hooper_index,score_total,confianza_score
count,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,...,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00
mean,288.64,289.80,188.11,194.25,120.81,119.79,87.37,86.50,166.58,167.98,...,1.01,162.42,163.31,29.55,71.18,2.33,1.18,15.45,21.06,63.30
std,134.80,139.13,84.62,87.83,58.21,53.95,45.90,44.87,71.78,71.53,...,0.99,36.27,35.20,8.15,11.77,2.23,1.61,4.76,10.24,16.26
min,80.00,80.00,50.00,50.00,40.00,40.00,25.00,25.00,50.00,50.00,...,0.00,62.50,52.80,18.00,45.00,0.00,0.00,5.00,0.00,24.00
25%,186.30,178.35,123.98,126.55,77.57,80.82,50.10,51.05,109.00,111.60,...,0.00,135.93,139.10,23.00,62.78,1.00,0.00,12.00,13.00,52.00
50%,261.90,257.45,168.55,181.25,107.00,110.60,80.40,79.30,153.75,153.85,...,1.00,162.30,163.00,29.00,70.70,2.00,1.00,15.00,20.00,64.00
75%,386.05,379.00,244.12,250.45,152.25,152.35,112.82,114.72,216.52,221.72,...,2.00,188.05,187.05,35.00,79.12,4.00,1.00,19.00,28.50,76.00
max,600.00,600.00,400.00,400.00,320.00,316.20,230.00,230.00,340.00,340.00,...,3.00,240.00,240.00,58.00,106.10,10.00,9.00,28.00,48.50,100.00


## Guardar datos

Ahora vamos a guardar el dataset generado en un archivo CSV. Este archivo será leído automáticamente por los siguientes notebooks, por lo que es importante ejecutar este paso correctamente.

El archivo se guardará en la carpeta `datos/sinteticos/` dentro del proyecto.

In [6]:
ruta_salida = proyecto_raiz / "datos" / "sinteticos" / "dataset_sintetico.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"Datos guardados en: {ruta_salida}")

Datos guardados en: /Users/__robeerr/Programacion_Local/IntApp v2/datos/sinteticos/dataset_sintetico.csv


## Resumen y siguiente paso

---

**Lo que hemos hecho en este notebook:**

- Generado un dataset sintético con **500 deportistas** y sus variables fisiológicas y de entrenamiento.
- Comprobado que los datos tienen una distribución realista de niveles de riesgo.
- Guardado el dataset en `datos/sinteticos/dataset_sintetico.csv`.

---

**Siguiente paso: ejecuta el notebook `02_exploracion_datos.ipynb`**

En ese notebook exploraremos el dataset en detalle: veremos qué variables están más relacionadas con el riesgo de lesión, detectaremos valores atípicos y prepararemos los datos para el modelo de machine learning.

---

> Si has modificado `N_DEPORTISTAS` o `SEMILLA`, recuerda volver a ejecutar todos los notebooks desde el principio para que los cambios se propaguen correctamente.